# Adapters Smoke Test (Colab GPU)

`voicesecure.evaluators.adapters` 의 5개 어댑터를 실제 GPU에서 한 번에 검증한다.
(XTTS는 별도 `xtts_smoke_test.ipynb`에서 검증함 — 여기서는 제외.)

대상:
| # | 어댑터 | 메서드 | 모델 | 모델 크기 |
|---|---|---|---|---|
| 1 | `CAMPlusAdapter` | `extract_embedding` | Wespeaker CAM++ (ONNX) | ~30 MB |
| 2 | `WavLMSVAdapter` | `extract_embedding` | microsoft/wavlm-base-plus-sv | ~380 MB |
| 3 | `Wav2Vec2KoreanAdapter` | `transcribe` | kresnik/wav2vec2-large-xlsr-korean | ~1.2 GB |
| 4 | `ECAPATDNNAdapter` | `extract_embedding` (+ grad) | speechbrain/spkrec-ecapa-voxceleb | ~80 MB |
| 5 | `CosyVoiceAdapter` | `clone`, `extract_embedding` | Fun-CosyVoice3-0.5B-2512 | ~수 GB |

각 셀에서:
- 의존성 설치 → 로드 → 더미 입력 호출 → shape/dtype/NaN/범위 assert
- 같은 입력 반복 호출의 결정성 확인 (해당되는 경우)

런타임: **GPU(T4) 필수**, 약 25-35분.


## 0. GPU 확인 + SDK clone

In [ ]:
!nvidia-smi | head -20

In [ ]:
%%bash
set -e
rm -rf voicesecure-sdk
git clone -q https://github.com/VoiceSecureHoseo/voicesecure-sdk.git
cd voicesecure-sdk
pip install -q -e .
echo "[OK] voicesecure-sdk installed"


In [ ]:
import os, sys
sys.path.insert(0, "/content/voicesecure-sdk/src")
os.chdir("/content/voicesecure-sdk")
print("cwd:", os.getcwd())


## 0-b. 공통 더미 입력 (16kHz mono float32 [-1, 1])

In [ ]:
import numpy as np
SR = 16000
def make_audio(freq=440.0, duration=3.0, sr=SR):
    t = np.arange(int(sr*duration), dtype=np.float32) / sr
    return (0.3 * np.sin(2*np.pi*freq*t)).astype(np.float32)

audio_a = make_audio(440.0)
audio_b = make_audio(880.0)
print("audio_a:", audio_a.shape, audio_a.dtype, "range:", (audio_a.min(), audio_a.max()))
print("audio_b:", audio_b.shape, audio_b.dtype)

# 실제 사람 목소리 ref.wav (CosyVoice clone 입력으로도 활용)
!wget -q -O /content/ref.wav https://github.com/coqui-ai/TTS/raw/dev/tests/data/ljspeech/wavs/LJ001-0001.wav
from voicesecure.utils.audio import load_audio
ref = load_audio("/content/ref.wav")
print("ref.wav:", ref.shape, "dur:", len(ref)/SR, "s")


## 1. CAMPlusAdapter (Wespeaker CAM++ ONNX)

의존성: `onnxruntime`, `huggingface-hub` (둘 다 SDK pyproject에 이미 있음).

In [ ]:
import time, numpy as np, torch
from voicesecure.evaluators.adapters import CAMPlusAdapter

t0 = time.time()
cam = CAMPlusAdapter()
print(f"[load CAM++] {time.time()-t0:.1f}s")

emb_a1 = cam.extract_embedding(audio_a)
emb_a2 = cam.extract_embedding(audio_a)
emb_b  = cam.extract_embedding(audio_b)

assert emb_a1.shape == (512,), emb_a1.shape
assert emb_a1.dtype == np.float32
assert np.isfinite(emb_a1).all()
assert np.allclose(emb_a1, emb_a2, atol=1e-5), "결정성 깨짐"
assert not np.allclose(emb_a1, emb_b, atol=1e-3), "다른 입력인데 동일"
print("[OK] CAM++ embedding shape", emb_a1.shape, "L2:", float(np.linalg.norm(emb_a1)))


## 2. WavLMSVAdapter (microsoft/wavlm-base-plus-sv)

의존성: `transformers`, `torch` (SDK 기본 의존성).

In [ ]:
import time
from voicesecure.evaluators.adapters import WavLMSVAdapter

t0 = time.time()
wavlm = WavLMSVAdapter()
print(f"[load WavLM-SV] {time.time()-t0:.1f}s")

emb_a1 = wavlm.extract_embedding(audio_a)
emb_a2 = wavlm.extract_embedding(audio_a)
emb_b  = wavlm.extract_embedding(audio_b)

assert emb_a1.ndim == 1
assert emb_a1.dtype == np.float32
assert emb_a1.size > 0
assert np.isfinite(emb_a1).all()
assert np.allclose(emb_a1, emb_a2, atol=1e-4), "결정성 깨짐"
assert not np.allclose(emb_a1, emb_b, atol=1e-3), "다른 입력인데 동일"
print("[OK] WavLM-SV embedding shape", emb_a1.shape, "L2:", float(np.linalg.norm(emb_a1)))


## 3. Wav2Vec2KoreanAdapter (kresnik/wav2vec2-large-xlsr-korean)

의존성: `transformers`. 모델 1.2 GB 다운로드라 가장 느림.
사인파는 음성이 아니라 빈 문자열에 가까운 텍스트가 나올 수 있음 — str 타입만 확인.

In [ ]:
import time
from voicesecure.evaluators.adapters import Wav2Vec2KoreanAdapter

t0 = time.time()
asr = Wav2Vec2KoreanAdapter()
print(f"[load wav2vec2] {time.time()-t0:.1f}s")

t0 = time.time()
text_a = asr.transcribe(audio_a)
text_ref = asr.transcribe(ref)
print(f"[transcribe x2] {time.time()-t0:.1f}s")
print("sine 440Hz ->", repr(text_a))
print("ref.wav     ->", repr(text_ref))

assert isinstance(text_a, str)
assert isinstance(text_ref, str)
print("[OK] wav2vec2 transcribe returned str")


## 4. ECAPATDNNAdapter (SpeechBrain spkrec-ecapa-voxceleb)

의존성: `speechbrain`. SDK pyproject에 없어서 셀에서 설치.

In [ ]:
!pip install -q speechbrain

In [ ]:
import time
from voicesecure.evaluators.adapters.ecapa_tdnn import ECAPATDNNAdapter

t0 = time.time()
ecapa = ECAPATDNNAdapter()
print(f"[load ECAPA-TDNN] {time.time()-t0:.1f}s")

emb_a1 = ecapa.extract_embedding(audio_a)
emb_a2 = ecapa.extract_embedding(audio_a)
emb_b  = ecapa.extract_embedding(audio_b)

assert emb_a1.shape == (192,), emb_a1.shape
assert emb_a1.dtype == np.float32
assert np.isfinite(emb_a1).all()
assert np.allclose(emb_a1, emb_a2, atol=1e-4), "결정성 깨짐"
assert not np.allclose(emb_a1, emb_b, atol=1e-3), "다른 입력인데 동일"
print("[OK] ECAPA-TDNN embedding shape", emb_a1.shape, "L2:", float(np.linalg.norm(emb_a1)))

# gradient 통과 검증 (FGSM 공격 호환성)
audio_t = torch.from_numpy(audio_a).float().unsqueeze(0).requires_grad_(True)
emb_grad = ecapa.extract_embedding_grad(audio_t)
loss = emb_grad.norm()
loss.backward()
assert audio_t.grad is not None
assert torch.isfinite(audio_t.grad).all()
print("[OK] ECAPA-TDNN gradient flows; grad_norm =", float(audio_t.grad.norm()))


## 5. CosyVoiceAdapter (FunAudioLLM/Fun-CosyVoice3-0.5B-2512)

가장 무거운 어댑터. 레포 clone + requirements 설치 + 모델 스냅샷 다운로드.
clone()과 extract_embedding() 둘 다 검증.

In [ ]:
%%bash
set -e
cd /content
if [ ! -d CosyVoice ]; then
  git clone -q --recursive https://github.com/FunAudioLLM/CosyVoice.git
fi
pip install -q -r CosyVoice/requirements.txt
echo "[OK] CosyVoice repo + deps installed"


In [ ]:
from huggingface_hub import snapshot_download
import os, time
t0 = time.time()
target = "/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B-2512"
if not os.path.isdir(target) or not os.listdir(target):
    snapshot_download("FunAudioLLM/Fun-CosyVoice3-0.5B-2512", local_dir=target)
size_mb = sum(os.path.getsize(os.path.join(d, f))
              for d, _, fs in os.walk(target) for f in fs) // (1024*1024)
print(f"[snapshot] {time.time()-t0:.1f}s, dir size: {size_mb} MB")


In [ ]:
import os, time
os.chdir("/content")  # CosyVoice 경로 일관성 위해 /content로 이동
from voicesecure.evaluators.adapters.cosyvoice import CosyVoiceAdapter

t0 = time.time()
cosy = CosyVoiceAdapter(
    model_dir="CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B-2512",
    cosyvoice_root="CosyVoice",
)
print(f"[load CosyVoice3] {time.time()-t0:.1f}s")


In [ ]:
# extract_embedding 검증 (CosyVoice 내장 CAM++)
emb_a1 = cosy.extract_embedding(audio_a)
emb_a2 = cosy.extract_embedding(audio_a)
emb_b  = cosy.extract_embedding(audio_b)

assert emb_a1.ndim == 1
assert emb_a1.dtype == np.float32
assert emb_a1.size in (192, 512), f"unexpected dim {emb_a1.size}"
assert np.isfinite(emb_a1).all()
assert np.allclose(emb_a1, emb_a2, atol=1e-3), "결정성 깨짐"
assert not np.allclose(emb_a1, emb_b, atol=1e-3), "다른 입력인데 동일"
print("[OK] CosyVoice extract_embedding shape", emb_a1.shape)


In [ ]:
# clone 검증 (실제 ref.wav로)
import time
t0 = time.time()
cloned = cosy.clone(ref, text="안녕하세요. 보이스시큐어 SDK 입니다.")
print(f"[clone] {time.time()-t0:.1f}s, cloned:", cloned.shape, cloned.dtype, "dur:", len(cloned)/SR, "s")

assert isinstance(cloned, np.ndarray)
assert cloned.ndim == 1
assert cloned.dtype == np.float32
assert np.isfinite(cloned).all()
assert cloned.min() >= -1.0 and cloned.max() <= 1.0
assert len(cloned) > SR * 0.5
print("[OK] CosyVoice clone assertions passed")

from IPython.display import Audio, display
print("ref")
display(Audio(ref, rate=SR))
print("CosyVoice clone")
display(Audio(cloned, rate=SR))


## 6. 요약

여기까지 모든 셀이 `[OK]`로 끝났다면 5개 어댑터(XTTS 포함 시 6개) 모두
실제 환경에서 정상 동작한다는 뜻이다.

각 어댑터의 evaluator 매핑:
- `CAMPlusAdapter`, `WavLMSVAdapter`, `ECAPATDNNAdapter`,
  `CosyVoiceAdapter.extract_embedding` → `SpeakerEvaluator` 의 `wavlm_model` / `cam_model`
- `XTTSAdapter`, `CosyVoiceAdapter.clone` → `TTSEvaluator` 의 `xtts_model`
- `Wav2Vec2KoreanAdapter` → `ASREvaluator` 의 `asr_model`

OpenVoice2 어댑터는 현재 SDK에 없음 — `TTSEvaluator` 의 `openvoice_model`
파라미터는 정의만 되어 있고 호출부는 주석 처리됨.


## 트러블슈팅

- **CAM++ ONNX 다운로드 실패**: HF rate-limit. `!huggingface-cli login` 후 재시도.
- **wav2vec2 OOM**: 사인파 3초는 OK. 더 긴 입력은 청크 분할 필요.
- **ECAPA-TDNN `speechbrain` 설치 충돌**: numpy 버전. `!pip install -q --no-deps speechbrain` 후 부족 deps만.
- **CosyVoice repo clone 시 submodule 누락**: `--recursive` 빼먹지 말기 (Matcha-TTS 필요).
- **CosyVoice 모델 다운로드가 ~1시간**: HF 네트워크 변동. `local_dir` 채워져 있으면 스킵.
- **CosyVoice import 실패 (`cosyvoice` 모듈 못 찾음)**: 셀이 `os.chdir("/content")` 하므로 cwd가 `/content`인지 확인.
